<a href="https://colab.research.google.com/github/depimuhammadr/Borrow_App/blob/main/kasus07_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UTS Intelegensi Buatan — Kasus 7 (Diabetes pada Wanita Pima)
**Mata Kuliah:** Intelegensi Buatan (3 SKS) — Universitas Teknologi Bandung  
**Dataset:** `kasus07_pima_diabetes.csv` (`target`: `type` -> `Yes` / `No`)

### Panduan Menjalankan (Tinggal Run Semua Sel):
1. Klik **Runtime -> Run all** (atau tekan `Ctrl + F9`).
2. Jika diminta mengunggah file pada Sel 2, pilih file `kasus07_pima_diabetes.csv` (atau biarkan terunduh otomatis dari repositori kuliah).
3. Sel 3–5 akan otomatis membuat file `fitur.py`, `latih.py`, dan `app.py`.
4. Sel 6 melatih model dan menyimpan artefak ke `models/model_diabetes_kasus07.joblib`.
5. Sel 7 menjalankan aplikasi **Streamlit** langsung di Google Colab.


In [1]:
!pip install -q scikit-learn pandas numpy joblib streamlit pyngrok


In [2]:
from pathlib import Path
import shutil
import urllib.request

for folder in ["dataset", "models", "notebook"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

data_dest = Path("dataset/kasus07_pima_diabetes.csv")

if Path("kasus07_pima_diabetes.csv").exists():
    shutil.move("kasus07_pima_diabetes.csv", data_dest)
    print("File lokal dipindahkan ke:", data_dest)

if not data_dest.exists():
    url = "https://raw.githubusercontent.com/Muhammad-Ikhwan-Fathulloh/Artificial-Intelegent-Course-Bank/main/UTS/dataset/kasus07_pima_diabetes.csv"
    try:
        urllib.request.urlretrieve(url, data_dest)
        print("Berhasil mengunduh dataset ke:", data_dest)
    except Exception as e:
        print("Gagal mengunduh otomatis:", e)
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            shutil.move(fname, data_dest)
            print("File diunggah ke:", data_dest)


Berhasil mengunduh dataset ke: dataset/kasus07_pima_diabetes.csv


In [3]:
%%writefile fitur.py
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

TARGET_COL = "type"
RAW_COLS = ["npreg", "glu", "bp", "skin", "bmi", "ped", "age"]

FEATURE_COLS = [
    "npreg",
    "glu",
    "bp",
    "skin",
    "bmi",
    "ped",
    "age",
    "rasio_glukosa_bmi",
    "interaksi_usia_hamil",
    "glukosa_tinggi",
]


def cek_kualitas_data(df: pd.DataFrame) -> dict:
    """Mengecek kondisi awal dataset sebelum dibagi train/test."""
    return {
        "jumlah_baris": int(df.shape[0]),
        "jumlah_kolom": int(df.shape[1]),
        "missing_value": df.isnull().sum().to_dict(),
        "duplikat": int(df.duplicated().sum()),
        "skin_tidak_wajar_gt_65": int((df["skin"] > 65).sum()),
        "distribusi_target": df[TARGET_COL].value_counts().to_dict(),
    }


class FiturDiabetes(BaseEstimator, TransformerMixin):
    """Membersihkan nilai tidak wajar medis & membuat fitur turunan."""

    def fit(self, X, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        data = X.copy()

        for col in RAW_COLS:
            data[col] = pd.to_numeric(data[col], errors="coerce")

        # Nilai skin > 65 (misal 99) dianggap tidak wajar -> jadikan NaN agar diimputasi median di pipeline
        data.loc[data["skin"] > 65, "skin"] = np.nan

        # Fitur turunan 1: gabungan glukosa & BMI
        data["rasio_glukosa_bmi"] = (data["glu"] * data["bmi"]) / 100.0

        # Fitur turunan 2: interaksi usia dan jumlah kehamilan (+1 agar npreg=0 tetap bermakna)
        data["interaksi_usia_hamil"] = data["age"] * (data["npreg"] + 1.0)

        # Fitur turunan 3: penanda glukosa >= 140 mg/dL (1 = ya, 0 = tidak)
        data["glukosa_tinggi"] = (data["glu"] >= 140).astype(float)

        return data[FEATURE_COLS]


def siapkan_xy(df: pd.DataFrame):
    """Memisahkan fitur (X) dan target biner (y: Yes=1, No=0)."""
    X = df[RAW_COLS].copy()
    y = df[TARGET_COL].map({"No": 0, "Yes": 1}).astype(int)
    return X, y


Writing fitur.py


In [4]:
%%writefile latih.py
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from fitur import FEATURE_COLS, FiturDiabetes, cek_kualitas_data, siapkan_xy

BASE_DIR = Path(__file__).resolve().parent
DATA_PATH = BASE_DIR / "dataset" / "kasus07_pima_diabetes.csv"
MODEL_PATH = BASE_DIR / "models" / "model_diabetes_kasus07.joblib"


def buat_preprocessor() -> ColumnTransformer:
    proses_angka = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    return ColumnTransformer(
        transformers=[("angka", proses_angka, FEATURE_COLS)],
        remainder="drop",
    )


def latih_model(data_path: Path = DATA_PATH, model_path: Path = MODEL_PATH) -> dict:
    df = pd.read_csv(data_path)
    info_audit = cek_kualitas_data(df)
    X, y = siapkan_xy(df)

    # 1. Split train & test di awal sebelum preprocessing
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    preprocessor = buat_preprocessor()
    kandidat = {
        "Logistic Regression": Pipeline(
            steps=[
                ("fitur", FiturDiabetes()),
                ("prep", preprocessor),
                ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
            ]
        ),
        "Random Forest": Pipeline(
            steps=[
                ("fitur", FiturDiabetes()),
                ("prep", preprocessor),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=200,
                        max_depth=5,
                        min_samples_leaf=3,
                        class_weight="balanced",
                        random_state=42,
                    ),
                ),
            ]
        ),
    }

    # 2. Cross-validation di data train saja
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_ringkasan = {}
    for nama, pipe in kandidat.items():
        skor = cross_validate(
            pipe,
            X_train,
            y_train,
            cv=cv,
            scoring={"roc_auc": "roc_auc", "f1": "f1", "recall": "recall", "accuracy": "accuracy"},
        )
        cv_ringkasan[nama] = {
            "cv_roc_auc": float(np.mean(skor["test_roc_auc"])),
            "cv_f1": float(np.mean(skor["test_f1"])),
            "cv_recall": float(np.mean(skor["test_recall"])),
            "cv_accuracy": float(np.mean(skor["test_accuracy"])),
        }

    # Pilih model dengan rata-rata ROC-AUC terbaik pada CV
    model_terbaik_nama = max(cv_ringkasan, key=lambda k: cv_ringkasan[k]["cv_roc_auc"])
    model_terbaik = kandidat[model_terbaik_nama]
    model_terbaik.fit(X_train, y_train)

    # 3. Evaluasi satu kali pada data test
    pred_test = model_terbaik.predict(X_test)
    proba_test = model_terbaik.predict_proba(X_test)[:, 1]

    metrik_test = {
        "accuracy": float(accuracy_score(y_test, pred_test)),
        "precision": float(precision_score(y_test, pred_test, zero_division=0)),
        "recall": float(recall_score(y_test, pred_test, zero_division=0)),
        "f1": float(f1_score(y_test, pred_test, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test, proba_test)),
        "confusion_matrix": confusion_matrix(y_test, pred_test).tolist(),
    }

    nilai_default = {col: float(X_train[col].median()) for col in X_train.columns}

    paket = {
        "pipeline": model_terbaik,
        "model_terbaik": model_terbaik_nama,
        "cv_ringkasan": cv_ringkasan,
        "metrik_test": metrik_test,
        "info_audit": info_audit,
        "nilai_default": nilai_default,
    }

    model_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(paket, model_path)
    return paket


if __name__ == "__main__":
    hasil = latih_model()
    print("Model terbaik:", hasil["model_terbaik"])
    print("Hasil CV (Train):", hasil["cv_ringkasan"])
    print("Metrik Test:", hasil["metrik_test"])


Writing latih.py


In [5]:
%%writefile app.py
from pathlib import Path
import joblib
import pandas as pd
import streamlit as st

from latih import MODEL_PATH, latih_model

st.set_page_config(page_title="UTS AI - Kasus 7", page_icon="🩺", layout="wide")


@st.cache_resource
def muat_model(path: Path):
    if not path.exists():
        return None
    return joblib.load(path)


st.title("🩺 Klasifikasi Risiko Diabetes — Kasus 7")
st.write("Aplikasi UTS Intelegensi Buatan (`kasus07_pima_diabetes.csv`)")

paket = muat_model(MODEL_PATH)
if paket is None:
    st.warning("Model belum ada. Klik tombol di bawah untuk melatih model.")
    if st.button("Latih Sekarang"):
        latih_model()
        st.cache_resource.clear()
        st.rerun()
    st.stop()

pipe = paket["pipeline"]
metrik = paket["metrik_test"]
cv_df = pd.DataFrame(paket["cv_ringkasan"]).T
default_val = paket["nilai_default"]

tab1, tab2 = st.tabs(["Prediksi Data Baru", "Metrik & Perbandingan Model"])

with tab1:
    st.subheader("Masukkan Data Pemeriksaan")
    c1, c2 = st.columns(2)
    with c1:
        npreg = st.number_input("Jumlah Kehamilan (npreg)", 0, 20, int(round(default_val["npreg"])))
        glu = st.number_input("Glukosa Plasma (glu)", 1, 300, int(round(default_val["glu"])))
        bp = st.number_input("Tekanan Darah Diastolik (bp)", 1, 200, int(round(default_val["bp"])))
        skin = st.number_input("Ketebalan Kulit Triceps (skin)", 1.0, 120.0, float(round(min(default_val["skin"], 65.0), 1)))
    with c2:
        bmi = st.number_input("BMI", 10.0, 80.0, float(round(default_val["bmi"], 1)))
        ped = st.number_input("Diabetes Pedigree (ped)", 0.01, 5.0, float(round(default_val["ped"], 3)))
        age = st.number_input("Usia (age)", 15, 110, int(round(default_val["age"])))

    if st.button("Prediksi", type="primary"):
        data_input = pd.DataFrame(
            [{"npreg": npreg, "glu": glu, "bp": bp, "skin": skin, "bmi": bmi, "ped": ped, "age": age}]
        )
        proba_yes = float(pipe.predict_proba(data_input)[0, 1])
        pred = int(pipe.predict(data_input)[0])
        label = "Yes" if pred == 1 else "No"

        st.success(f"Hasil Prediksi: **{label}**")
        st.metric("Probabilitas Diabetes (`Yes`)", f"{proba_yes * 100:.2f}%")

with tab2:
    st.subheader(f"Model Terpilih: {paket['model_terbaik']}")
    k1, k2, k3, k4, k5 = st.columns(5)
    k1.metric("Accuracy", f"{metrik['accuracy']:.4f}")
    k2.metric("Precision", f"{metrik['precision']:.4f}")
    k3.metric("Recall", f"{metrik['recall']:.4f}")
    k4.metric("F1-Score", f"{metrik['f1']:.4f}")
    k5.metric("ROC-AUC", f"{metrik['roc_auc']:.4f}")

    st.markdown("**Confusion Matrix (`[[TN, FP], [FN, TP]]`)**")
    st.write(metrik["confusion_matrix"])

    st.markdown("**Perbandingan 5-Fold Cross-Validation (Data Train)**")
    st.dataframe(cv_df.style.format("{:.4f}"), use_container_width=True)


Writing app.py


In [6]:
!python latih.py


Model terbaik: Random Forest
Hasil CV (Train): {'Logistic Regression': {'cv_roc_auc': 0.8301731601731603, 'cv_f1': 0.6962932719454459, 'cv_recall': 0.72, 'cv_accuracy': 0.7875}, 'Random Forest': {'cv_roc_auc': 0.8558441558441559, 'cv_f1': 0.6585443541965281, 'cv_recall': 0.6454545454545455, 'cv_accuracy': 0.775}}
Metrik Test: {'accuracy': 0.75, 'precision': 0.625, 'recall': 0.7142857142857143, 'f1': 0.6666666666666666, 'roc_auc': 0.7802197802197802, 'confusion_matrix': [[20, 6], [4, 10]]}


In [ ]:
import subprocess
import time
import urllib.request

subprocess.run(["pkill", "-f", "streamlit"], check=False)

proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(3)

# Menggunakan cloudflared (tanpa perlu daftar/token) agar aplikasi Streamlit langsung bisa dibuka
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8501


2026-11-22T14:01:48Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-22T14:01:48Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-22T14:01:52Z INF +--------------------------------------------------------------------------------------------+
2026-11-22T14:01:52Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-22T14:01:52Z INF |  https://luis-lamp-wife-alias.trycloudflare.com       